In [1]:
import os
import glob
import json
import sys
import time
import requests
from datetime import datetime

import pandas as pd
import numpy as np
import rasterio
import rasterstats
import rioxarray as rxr
import geopandas as gpd
from osgeo import gdal
from shapely.geometry import box
from tqdm.notebook import tqdm_notebook as tqdm

In [2]:
dir = os.getcwd()

In [4]:
# df = pd.read_csv(os.path.join(dir, 'datasets/South_VNM_3per_sampleID.csv'))
df = pd.read_csv(os.path.join(dir, 'datasets\South_sample_5x10000.csv'))
df

<>:2: SyntaxWarning: invalid escape sequence '\S'
<>:2: SyntaxWarning: invalid escape sequence '\S'
C:\Users\sonle\AppData\Local\Temp\ipykernel_14488\445749686.py:2: SyntaxWarning: invalid escape sequence '\S'
  df = pd.read_csv(os.path.join(dir, 'datasets\South_sample_5x10000.csv'))


,id,lon,lat,sample_1,sample_2,sample_3,sample_4,sample_5
0,518582,105.34409,9.759747,0,0,0,0,1
1,306736,105.16891,10.705672,0,0,0,0,1
2,515851,105.34139,9.483066,0,0,0,0,1
3,398571,105.24168,10.394855,0,0,0,0,1
4,807982,105.56148,10.331973,0,0,0,0,1
...,...,...,...,...,...,...,...,...
49391,664345,105.46446,9.630389,0,0,0,1,0
49392,1527014,106.54154,10.091225,0,0,0,1,0
49393,967309,105.67736,9.376165,0,0,1,0,0
49394,578874,105.39799,10.594281,0,1,0,0,0


In [7]:
sample = df[df['sample_1'] == 1]
sample

,id,lon,lat,sample_1,sample_2,sample_3,sample_4,sample_5
406,1151467,105.83366,9.403115,1,0,0,0,1
494,817216,105.56776,10.552958,1,0,0,0,1
570,1413880,106.17323,10.038224,1,0,0,0,1
769,582335,105.40158,9.667220,1,0,1,0,1
1165,988863,105.69353,9.341131,1,0,0,0,1
...,...,...,...,...,...,...,...,...
49375,879837,105.61268,10.075055,1,0,0,0,0
49380,755053,105.52554,9.334844,1,0,0,0,0
49381,1412410,106.16874,10.920370,1,0,0,0,0
49382,426142,105.26414,10.153209,1,0,0,0,0


In [14]:
api_key = '73f784c939b7c0017b23be2e72b482cc'

lats = sample['lat'].to_list()
lons = sample['lon'].to_list()

def create_json(lat:str, lon:str, start: int, end: int) -> dict:
    hist_url = f'http://api.openweathermap.org/data/2.5/air_pollution/history?lat={lat}&lon={lon}&start={start}&end={end}&appid={api_key}'
    r = requests.get(hist_url)
    json_out = r.json()

    return json_out

for i in tqdm(range(len(lats))[299:]):

    print(lats[i], lons[i])
    

    start_1 = int(time.mktime(datetime(2022, 1, 1).timetuple()))
    end_1 = int(time.mktime(datetime(2022, 12, 31).timetuple()))
   
    # Dumps json file
    json_1 = create_json(lats[i], lons[i], start_1, end_1)
    with open(f"datasets/owm/{lats[i]}_{lons[i]}_2022.json", "w") as outfile: 
        json.dump(json_1, outfile)




  0%|          | 0/9701 [00:00<?, ?it/s]

10.08853 105.36744
10.802691 106.20287
10.065173 106.64934
10.017563 105.46895
10.685011 105.94057
10.455043 105.80133
9.6061344 105.389
9.0985861 104.98296
10.302329 104.92547
9.8747311 105.36115
10.269989 105.42044
9.9843254 105.26414
10.879946 105.47884
9.8648491 105.51746
10.359821 104.83295
10.76586 105.65849
10.075055 105.43122
9.8370018 105.37643
10.592484 105.10154
10.329278 105.13029
9.6124229 105.88487
10.29065 106.71312
9.6501522 105.88307
10.907793 105.61538
10.215192 105.00812
10.42989 104.67484
10.370601 104.78623
9.6896782 105.97201
10.769453 105.2273
9.7687292 105.56328
10.863776 105.26144
10.175667 105.36744
9.6779995 105.46895
10.040021 105.56507
9.7893906 105.96123
10.441567 106.55502
10.432585 104.62543
10.319397 104.95422
10.561942 105.0701
10.680519 105.03416
10.948217 106.57298
10.862878 105.52016
10.42989 105.35217
10.236752 105.67646
10.439771 105.71599
9.6636267 105.93787
9.9861221 105.35397
10.181056 105.09615
10.367906 105.68993
10.912285 106.38434
10.07775 

In [12]:
json_1

{'cod': 401,
 'message': 'Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.'}

In [44]:
with open('JAXA_subtiles.json', 'r') as json_file:
    jaxa_subtiles = json.load(json_file)

tiles = list(jaxa_subtiles.keys())[1:16]
tiles.extend(list(jaxa_subtiles.keys())[33:])

bbox = [jaxa_subtiles[tile] for tile in tiles]

bbox = [box(tile[0], tile[1], tile[2], tile[3]) for tile in bbox]

tiles_gdf = gpd.GeoDataFrame({'tile': tiles, 
                              'geometry': bbox})
tiles_gdf = tiles_gdf.set_crs(epsg=4326)
tiles_gdf.head()
tiles_gdf.to_file('chosen_tiles.json', driver='GeoJSON', index=False)

In [35]:
tiles_gdf.crs

In [38]:
dist_gdf = gpd.read_file('gadm41_VNM_2.json')
dist_gdf = dist_gdf.to_crs(tiles_gdf.crs)
dist_gdf.head()

,GID_2,GID_0,COUNTRY,GID_1,NAME_1,NL_NAME_1,NAME_2,VARNAME_2,NL_NAME_2,TYPE_2,ENGTYPE_2,CC_2,HASC_2,geometry
0,VNM.1.1_1,VNM,Vietnam,VNM.1_1,AnGiang,NA,AnPhú,AnPhu,NA,Huyện,District,NA,VN.TT.AL,"MULTIPOLYGON (((105.15660 10.82870, 105.16010 ..."
1,VNM.1.3_1,VNM,Vietnam,VNM.1_1,AnGiang,NA,ChâuĐốc,ChauDoc,NA,Thànhphố,City,NA,VN.KG.AB,"MULTIPOLYGON (((105.10500 10.63830, 105.08150 ..."
2,VNM.1.4_1,VNM,Vietnam,VNM.1_1,AnGiang,NA,ChâuPhú,ChauPhu,NA,Huyện,District,NA,VN.HP.AD,"MULTIPOLYGON (((105.18790 10.46030, 105.17980 ..."
3,VNM.1.5_1,VNM,Vietnam,VNM.1_1,AnGiang,NA,ChâuThành,ChauThanh,NA,Huyện,District,NA,VN.GL.AK,"MULTIPOLYGON (((105.37090 10.36220, 105.36820 ..."
4,VNM.1.2_1,VNM,Vietnam,VNM.1_1,AnGiang,NA,ChợMới,ChoMoi,NA,Huyện,District,NA,VN.BD.AL,"MULTIPOLYGON (((105.48720 10.53950, 105.48840 ..."


In [78]:
dist_geom = [geom for geom in dist_gdf.geometry]

intersects_geom = []
for geom in dist_geom:
    gdf = tiles_gdf.iloc[tiles_gdf.sindex.query(geom, predicate='intersects')]
    if len(gdf) > 0:
        intersects_geom.append(geom)

In [98]:
intersects_geom = list(set(intersects_geom))
dist_filtered = dist_gdf[dist_gdf.geometry.isin(intersects_geom)]

In [99]:
dist_filtered['lon'] = dist_filtered.centroid.x
dist_filtered['lat'] = dist_filtered.centroid.y
dist_filtered = dist_filtered[['GID_2', 'lat', 'lon']]
dist_dict = dist_filtered.set_index('GID_2').to_dict('index')
dist_dict

C:\Users\sonle\AppData\Local\Temp\ipykernel_28392\439494382.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  dist_filtered['lon'] = dist_filtered.centroid.x
c:\Users\sonle\anaconda3\envs\geospatial\Lib\site-packages\geopandas\geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
C:\Users\sonle\AppData\Local\Temp\ipykernel_28392\439494382.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  dist_filtered['lat'] = dist_filte

{'VNM.1.1_1': {'lat': 10.846404435922919, 'lon': 105.10074651097067},
 'VNM.1.3_1': {'lat': 10.674122054506455, 'lon': 105.08741946173457},
 'VNM.1.4_1': {'lat': 10.557603596145857, 'lon': 105.1797822230923},
 'VNM.1.5_1': {'lat': 10.421303385083814, 'lon': 105.26575383802694},
 'VNM.1.2_1': {'lat': 10.474944476027188, 'lon': 105.46144105444661},
 'VNM.1.6_1': {'lat': 10.371310030995655, 'lon': 105.42804673733518},
 'VNM.1.7_1': {'lat': 10.654870714342247, 'lon': 105.2757414511293},
 'VNM.1.9_1': {'lat': 10.800174081298888, 'lon': 105.18645918433309},
 'VNM.1.10_1': {'lat': 10.29683597866068, 'lon': 105.25556484274274},
 'VNM.1.8_1': {'lat': 10.547266352717338, 'lon': 105.01170710864969},
 'VNM.1.11_1': {'lat': 10.412558176021932, 'lon': 104.95909074038255},
 'VNM.7.5_1': {'lat': 10.5767716473971, 'lon': 107.08808543963191},
 'VNM.3.1_1': {'lat': 21.27452053546984, 'lon': 106.19658531724376},
 'VNM.3.2_1': {'lat': 21.324537974828658, 'lon': 105.96767319204977},
 'VNM.3.3_1': {'lat': 21

In [126]:
list(dist_dict.keys()).index('VNM.44.10_1')

290

In [127]:
list(dist_dict.keys())[290:]

['VNM.44.10_1',
 'VNM.44.11_1',
 'VNM.44.12_1',
 'VNM.44.13_1',
 'VNM.46.8_1',
 'VNM.49.1_1',
 'VNM.49.2_1',
 'VNM.49.3_1',
 'VNM.49.4_1',
 'VNM.49.5_1',
 'VNM.49.6_1',
 'VNM.49.7_1',
 'VNM.49.8_1',
 'VNM.49.9_1',
 'VNM.49.10_1',
 'VNM.49.11_1',
 'VNM.49.12_1',
 'VNM.49.13_1',
 'VNM.49.14_1',
 'VNM.51.1_1',
 'VNM.51.3_1',
 'VNM.51.4_1',
 'VNM.51.5_1',
 'VNM.51.6_1',
 'VNM.51.7_1',
 'VNM.51.8_1',
 'VNM.51.9_1',
 'VNM.51.10_1',
 'VNM.51.11_1',
 'VNM.53.9_1',
 'VNM.55.1_1',
 'VNM.55.2_1',
 'VNM.55.3_1',
 'VNM.55.4_1',
 'VNM.55.5_1',
 'VNM.55.6_1',
 'VNM.55.7_1',
 'VNM.55.8_1',
 'VNM.56.1_1',
 'VNM.56.2_1',
 'VNM.56.3_1',
 'VNM.56.4_1',
 'VNM.56.5_1',
 'VNM.56.6_1',
 'VNM.56.7_1',
 'VNM.56.8_1',
 'VNM.56.9_1',
 'VNM.57.2_1',
 'VNM.57.1_1',
 'VNM.57.3_1',
 'VNM.57.4_1',
 'VNM.57.6_1',
 'VNM.57.5_1',
 'VNM.57.7_1',
 'VNM.57.8_1',
 'VNM.57.11_1',
 'VNM.57.10_1',
 'VNM.57.12_1',
 'VNM.57.13_1',
 'VNM.57.14_1',
 'VNM.57.16_1',
 'VNM.57.17_1',
 'VNM.57.15_1',
 'VNM.57.18_1',
 'VNM.57.19_1',
 'VN

In [128]:
api_key = 'ac31e02046794bd3f959fbc98fc25231'

def create_json(location_name:str, start: int, end: int) -> dict:
    global dist_dict
    lat = dist_dict[location_name]['lat']
    lon = dist_dict[location_name]['lon']
    hist_url = f'http://api.openweathermap.org/data/2.5/air_pollution/history?lat={lat}&lon={lon}&start={start}&end={end}&appid={api_key}'
    r = requests.get(hist_url)
    json_out = r.json()
    json_out['name'] = location_name

    return json_out

for location_name in list(dist_dict.keys())[290:]:

    print(location_name)
    
    # 6 month extraction to avoid request timeout from OWM
    # loc_dict = {}

    for year in list(range(2018, 2023, 1)):
        print('     ', year)
        # January - June 
        start_1 = int(time.mktime(datetime(year, 1, 1).timetuple()))
        end_1 = int(time.mktime(datetime(year, 6, 30).timetuple()))

        # Dumps json file
        json_1 = create_json(location_name, start_1, end_1)
        with open(f"owm_pollution/HN_{location_name}_16_{year}.json", "w") as outfile: 
            json.dump(json_1, outfile)

        # July - December 
        start_2 = int(time.mktime(datetime(year, 7, 1).timetuple()))
        end_2 = int(time.mktime(datetime(year, 12, 31).timetuple()))

        # Dumps json file
        json_2 = create_json(location_name, start_2, end_2)
        with open(f"owm_pollution/HN_{location_name}_712_{year}.json", "w") as outfile: 
            json.dump(json_2, outfile)

        # full_json = json_1 | json_2
    
    # loc_dict.update(full_json)



VNM.44.10_1
      2018
      2019
      2020
      2021
      2022
VNM.44.11_1
      2018
      2019
      2020
      2021
      2022
VNM.44.12_1
      2018
      2019
      2020
      2021
      2022
VNM.44.13_1
      2018
      2019
      2020
      2021
      2022
VNM.46.8_1
      2018
      2019
      2020
      2021
      2022
VNM.49.1_1
      2018
      2019
      2020
      2021
      2022
VNM.49.2_1
      2018
      2019
      2020
      2021
      2022
VNM.49.3_1
      2018
      2019
      2020
      2021
      2022
VNM.49.4_1
      2018
      2019
      2020
      2021
      2022
VNM.49.5_1
      2018
      2019
      2020
      2021
      2022
VNM.49.6_1
      2018
      2019
      2020
      2021
      2022
VNM.49.7_1
      2018
      2019
      2020
      2021
      2022
VNM.49.8_1
      2018
      2019
      2020
      2021
      2022
VNM.49.9_1
      2018
      2019
      2020
      2021
      2022
VNM.49.10_1
      2018
      2019
      2020
      2021
      2022
VNM.4

In [112]:
len(loc_dict['list'])

4297

In [4]:
paths = glob.glob('D:\Work\ADB\SAR\datasets\owm/*.json')
paths.sort()
paths

<>:1: SyntaxWarning: invalid escape sequence '\W'
<>:1: SyntaxWarning: invalid escape sequence '\W'
C:\Users\sonle\AppData\Local\Temp\ipykernel_38680\440931412.py:1: SyntaxWarning: invalid escape sequence '\W'
  paths = glob.glob('D:\Work\ADB\SAR\datasets\owm/*.json')


['D:\\Work\\ADB\\SAR\\datasets\\owm\\10.000495_105.14197_2022.json',
 'D:\\Work\\ADB\\SAR\\datasets\\owm\\10.000495_105.18778_2022.json',
 'D:\\Work\\ADB\\SAR\\datasets\\owm\\10.000495_105.2309_2022.json',
 'D:\\Work\\ADB\\SAR\\datasets\\owm\\10.000495_105.25874_2022.json',
 'D:\\Work\\ADB\\SAR\\datasets\\owm\\10.000495_105.30186_2022.json',
 'D:\\Work\\ADB\\SAR\\datasets\\owm\\10.000495_105.35307_2022.json',
 'D:\\Work\\ADB\\SAR\\datasets\\owm\\10.000495_105.41955_2022.json',
 'D:\\Work\\ADB\\SAR\\datasets\\owm\\10.000495_105.53363_2022.json',
 'D:\\Work\\ADB\\SAR\\datasets\\owm\\10.000495_105.56058_2022.json',
 'D:\\Work\\ADB\\SAR\\datasets\\owm\\10.000495_105.57764_2022.json',
 'D:\\Work\\ADB\\SAR\\datasets\\owm\\10.000495_105.72587_2022.json',
 'D:\\Work\\ADB\\SAR\\datasets\\owm\\10.000495_105.96303_2022.json',
 'D:\\Work\\ADB\\SAR\\datasets\\owm\\10.000495_105.97111_2022.json',
 'D:\\Work\\ADB\\SAR\\datasets\\owm\\10.000495_105.98279_2022.json',
 'D:\\Work\\ADB\\SAR\\datasets\\owm

In [43]:
dfs = []

for path in tqdm(paths):
    with open(path, 'r') as f:
        d = json.load(f)
    dates = []
    pm25s = []

    for i in range(len(d['list'])):
        dates.append(d['list'][i]['dt'])
        pm25s.append(d['list'][i]['components']['pm2_5'])


    lat_l = path.rindex('\\')
    lat_r = path.index('_')
    lon_r = path.rindex('_')

    df_p = pd.DataFrame({'lat': path[lat_l+1:lat_r],
                        'lon': path[lat_r+1:lon_r],
                        'dt': dates,
                        'pm25': pm25s})
    df_p['dt'] = pd.to_datetime(df_p['dt'], unit='s')
    df_p['dt'] = df_p['dt'].dt.tz_localize('UTC').dt.tz_convert('Asia/Saigon')
    df_p['month'] = df_p['dt'].dt.month
    df_p = df_p.groupby(['lat', 'lon', 'month']).agg({'pm25': 'mean'}).reset_index()

    df_p['lat'] = df_p['lat'].astype('float64')
    df_p['lon'] = df_p['lon'].astype('float64')

    dfs.append(df_p)

  0%|          | 0/48010 [00:00<?, ?it/s]

In [44]:
df_all = pd.concat(dfs, axis=0).reset_index(drop=True)
df_all

,lat,lon,month,pm25
0,10.000495,105.14197,1,30.825722
1,10.000495,105.14197,2,24.829599
2,10.000495,105.14197,3,25.805780
3,10.000495,105.14197,4,35.481083
4,10.000495,105.14197,5,15.307204
...,...,...,...,...
576115,9.999597,105.98009,8,30.194556
576116,9.999597,105.98009,9,21.298597
576117,9.999597,105.98009,10,56.314220
576118,9.999597,105.98009,11,42.334556


In [45]:
df_out = pd.merge(df, df_all, on=['lat', 'lon'], how='right')
df_out

,id,lon,lat,month,pm25
0,273937,105.14197,10.000495,1,30.825722
1,273937,105.14197,10.000495,2,24.829599
2,273937,105.14197,10.000495,3,25.805780
3,273937,105.14197,10.000495,4,35.481083
4,273937,105.14197,10.000495,5,15.307204
...,...,...,...,...,...
576115,1301298,105.98009,9.999597,8,30.194556
576116,1301298,105.98009,9.999597,9,21.298597
576117,1301298,105.98009,9.999597,10,56.314220
576118,1301298,105.98009,9.999597,11,42.334556


In [46]:
df_out.to_csv('datasets/owm_pm25_2022.csv', index=False)